# PlookCode — เทรนบน Google Colab (T4 ฟรี)

โยกการเทรนจากคอมบ้าน (GTX 1650, 4GB) ขึ้น Colab T4 (16GB) — เร็วกว่าหลายเท่า.

**เตรียมของใน Google Drive ก่อน 1 ครั้ง** — สร้างโฟลเดอร์ `MyDrive/localbrain-workspace/`
แล้วอัปโหลด dataset ที่สร้างไว้แล้วจากคอมบ้าน (ไม่ต้องขน raw parquet):

- `dataset-code-big/`  (~615MB — สำหรับ pretrain)
- `dataset-anneal/`    (~155MB — สำหรับ anneal)
- `dataset-english-sft/` (~71MB — SFT รวมเลขแล้ว)

วางให้โครงเป็น `MyDrive/localbrain-workspace/dataset-code-big/...` ฯลฯ

> Colab ฟรีตัดทุก ~12 ชม. — ไม่เป็นไร checkpoint เซฟลง Drive, รัน cell 1-6 ซ้ำแล้ว **resume เอง**.

**รันตามลำดับ cell 1 → 6**


## 1. เช็ค GPU (ต้องเป็น T4) — ถ้าไม่ใช่: Runtime ▸ Change runtime type ▸ T4 GPU


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


## 2. เมานต์ Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_WS = '/content/drive/MyDrive/localbrain-workspace'
import os; os.makedirs(DRIVE_WS, exist_ok=True)
print('workspace ->', DRIVE_WS)


## 3. โคลน repo + ติดตั้ง
repo เป็น private — ใส่ GitHub token ตอนถาม (Personal Access Token, สิทธิ์ repo). token ไม่ถูกเซฟในไฟล์.


In [ ]:
import getpass, os
REPO = 'YamikoZz/PlookCode'
if not os.path.exists('/content/localbrain'):
    tok = getpass.getpass('GitHub token: ')
    !git clone https://{tok}@github.com/{REPO}.git /content/localbrain
else:
    !cd /content/localbrain && git pull
%cd /content/localbrain
!pip -q install -e .


## 4. ลิงก์ workspace ไป Drive + เช็ค dataset ครบไหม
ทุกอย่างใน `workspace/` ชี้ไป Drive → checkpoint รอดตอนโดนตัด.


In [ ]:
import os
if os.path.islink('/content/localbrain/workspace') or os.path.exists('/content/localbrain/workspace'):
    !rm -rf /content/localbrain/workspace
os.symlink(DRIVE_WS, '/content/localbrain/workspace')

need = ['dataset-code-big', 'dataset-anneal', 'dataset-english-sft']
missing = [d for d in need if not os.path.exists(f'{DRIVE_WS}/{d}/meta.json')]
if missing:
    print('⚠️  ยังขาด dataset ใน Drive:', missing)
    print('    อัปโหลดโฟลเดอร์พวกนี้ไป', DRIVE_WS, 'ก่อน แล้วรัน cell นี้ใหม่')
else:
    print('✅ dataset ครบ พร้อมเทรน')


## 5. เริ่มเทรน (pretrain → anneal → SFT → eval)
ใช้ config T4 (batch 32, vram 14GB). รันนี้ค้างยาว — เซฟทุก 200 step ลง Drive.

**โดนตัด/หลุด?** ต่อ runtime ใหม่ แล้วรัน cell 1-5 ซ้ำ — มัน resume จาก checkpoint ล่าสุดใน Drive เอง.

> อยากเทรน **ต่อจากงานที่บ้าน** (เก็บ step ที่ทำไปแล้ว) แทนที่จะเริ่มใหม่?
> อัปโฟลเดอร์ `run-code-pretrain-big/` จากบ้านไป Drive ด้วย แล้ว**ลบบรรทัด** `os.environ['LOCALBRAIN_PRETRAIN_CONFIG']`
> ข้างล่าง (ให้ใช้ config เดิม batch 8 ไม่งั้น resume จะ error เพราะ batch ไม่ตรง).


In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['LOCALBRAIN_PRETRAIN_CONFIG'] = 'configs/code-pretrain-colab.json'  # ลบบรรทัดนี้ถ้าจะ resume งานบ้าน
os.environ['LOCALBRAIN_SFT_CONFIG'] = 'configs/code-sft-colab.json'  # SFT ใช้ VRAM ของ T4 ได้เต็ม
!python scripts/retrain.py


## 6. (ระหว่างเทรน) ดู log / ความคืบหน้า — รันใน cell ใหม่ได้ตลอด


In [ ]:
!tail -n 25 /content/localbrain/workspace/retrain.log
print('-'*50)
!tail -n 5 /content/localbrain/workspace/run-code-pretrain-big/metrics.jsonl 2>/dev/null


## 7. ลองคุยกับโมเดล (หลัง SFT เสร็จ)
โมเดลอยู่ที่ `workspace/run-english-sft2/` — เซฟใน Drive แล้ว ดาวน์โหลดกลับคอมบ้านมาใช้ต่อได้.


In [ ]:
import torch, sys
sys.argv = ['x']
from localbrain.coder import CodeModel
m = CodeModel('workspace/run-english-sft2')
for q in ['What is 7 + 8?', 'Write a Python function that returns if a number is even', 'hello']:
    print('Q:', q)
    print('A:', m.sample(q, n=1, greedy=True, max_tokens=120)[0])
    print('-'*50)
